In [2]:
from pathlib import Path
from rdflib import Graph

# Option A: local repository folder (recommended once you have the files locally)
REPO = Path(".")

# Option B: straight from GitHub (used automatically if the local files are missing)
RAW = "https://raw.githubusercontent.com/d-mau-m/fair_group_3/refs/heads/main/"

def src(rel):
    p = REPO / rel
    return str(p) if p.exists() else RAW + rel

data   = Graph().parse(src("data/rdf/artists.ttl"), format="turtle")
shapes = Graph().parse(src("shapes/artist-shapes.ttl"), format="turtle")
print(f"data: {len(data):,} triples · shapes: {len(shapes):,} triples")

data: 35,842 triples · shapes: 89 triples


In [3]:
from pyshacl import validate
conforms, report_graph, report_text = validate(data, shacl_graph=shapes, advanced=True, inference="none")
print("Conforms:", conforms)

Conforms: False


In [4]:
# Readable summary: one row per violation
import pandas as pd
from rdflib.namespace import SH
rows = []
for r in report_graph.subjects(SH.resultSeverity, None):
    rows.append({
        "focus node": str(report_graph.value(r, SH.focusNode)),
        "path": str(report_graph.value(r, SH.resultPath) or ""),
        "message": str(report_graph.value(r, SH.resultMessage)),
    })
violations = pd.DataFrame(rows)
print(len(violations), "violations")
violations.groupby("message").size().sort_values(ascending=False)

13 violations


message
More than 1 values on wd:Q1009324->schema1:containedInPlace    1
More than 1 values on wd:Q1044808->schema1:containedInPlace    1
More than 1 values on wd:Q205905->schema1:containedInPlace     1
More than 1 values on wd:Q2132->schema1:containedInPlace       1
More than 1 values on wd:Q239->schema1:containedInPlace        1
More than 1 values on wd:Q49186->schema1:containedInPlace      1
More than 1 values on wd:Q49218->schema1:containedInPlace      1
More than 1 values on wd:Q6084155->schema1:containedInPlace    1
More than 1 values on wd:Q6106->schema1:containedInPlace       1
More than 1 values on wd:Q65->schema1:containedInPlace         1
More than 1 values on wd:Q70->schema1:containedInPlace         1
More than 1 values on wd:Q79657->schema1:containedInPlace      1
More than 1 values on wd:Q79848->schema1:containedInPlace      1
dtype: int64

In [5]:
violations.sort_values(["message", "focus node"]).head(30)

,focus node,path,message
4,http://www.wikidata.org/entity/Q1009324,http://schema.org/containedInPlace,More than 1 values on wd:Q1009324->schema1:con...
12,http://www.wikidata.org/entity/Q1044808,http://schema.org/containedInPlace,More than 1 values on wd:Q1044808->schema1:con...
7,http://www.wikidata.org/entity/Q205905,http://schema.org/containedInPlace,More than 1 values on wd:Q205905->schema1:cont...
8,http://www.wikidata.org/entity/Q2132,http://schema.org/containedInPlace,More than 1 values on wd:Q2132->schema1:contai...
0,http://www.wikidata.org/entity/Q239,http://schema.org/containedInPlace,More than 1 values on wd:Q239->schema1:contain...
2,http://www.wikidata.org/entity/Q49186,http://schema.org/containedInPlace,More than 1 values on wd:Q49186->schema1:conta...
1,http://www.wikidata.org/entity/Q49218,http://schema.org/containedInPlace,More than 1 values on wd:Q49218->schema1:conta...
9,http://www.wikidata.org/entity/Q6084155,http://schema.org/containedInPlace,More than 1 values on wd:Q6084155->schema1:con...
11,http://www.wikidata.org/entity/Q6106,http://schema.org/containedInPlace,More than 1 values on wd:Q6106->schema1:contai...
3,http://www.wikidata.org/entity/Q65,http://schema.org/containedInPlace,More than 1 values on wd:Q65->schema1:containe...


In [6]:
# Save the full report as evidence for the report
Path("validation").mkdir(exist_ok=True)
Path("validation/shacl-report.txt").write_text(report_text, encoding="utf-8")
violations.to_csv("validation/shacl-violations.csv", index=False)
print("saved validation/shacl-report.txt and validation/shacl-violations.csv")

saved validation/shacl-report.txt and validation/shacl-violations.csv


In [7]:
import urllib.request
def read(rel):
    s = src(rel)
    return Path(s).read_text(encoding="utf-8") if not s.startswith("http") else urllib.request.urlopen(s).read().decode()

def run(rel, limit=10):
    q = read(rel)
    res = data.query(q)
    df = pd.DataFrame([[str(v) if v is not None else None for v in row] for row in res], columns=[str(v) for v in res.vars])
    print(f"{rel}: {len(df)} rows")
    return df.head(limit)

In [8]:
run("queries/cq1-born-in-netherlands.rq")

queries/cq1-born-in-netherlands.rq: 35 rows


,artist,name,birthPlace
0,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Alma-Tadema, Sir Lawrence",http://www.wikidata.org/entity/Q578267
1,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Appel, Karel",http://www.wikidata.org/entity/Q727
2,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Bauer, Marius",http://www.wikidata.org/entity/Q36600
3,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Blom, Ansuya",http://www.wikidata.org/entity/Q749
4,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Bock, Théophile de",http://www.wikidata.org/entity/Q36600
5,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Boumans, Bart",http://www.wikidata.org/entity/Q10001
6,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Brammer, Leonard",http://www.wikidata.org/entity/Q690
7,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Brouwn, Stanley",http://www.wikidata.org/entity/Q55
8,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Cleijne, Edgar",http://www.wikidata.org/entity/Q9832
9,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Collier, Edward",http://www.wikidata.org/entity/Q40844


In [9]:
run("queries/cq2-gender-per-century.rq", limit=20)

queries/cq2-gender-per-century.rq: 10 rows


,century,gender,artists
0,15,http://schema.org/Male,1
1,16,http://schema.org/Male,17
2,17,http://schema.org/Female,1
3,17,http://schema.org/Male,63
4,18,http://schema.org/Female,13
5,18,http://schema.org/Male,377
6,19,http://schema.org/Female,94
7,19,http://schema.org/Male,904
8,20,http://schema.org/Female,404
9,20,http://schema.org/Male,1508


In [10]:
run("queries/cq3-different-birth-death-country.rq")

queries/cq3-different-birth-death-country.rq: 443 rows


,artist,name,birthCountry,deathCountry
0,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Abbey, Edwin Austin",http://www.wikidata.org/entity/Q30,http://www.wikidata.org/entity/Q145
1,https://w3id.org/FAIR-course-UT/2025-2026/grou...,Absalon,http://www.wikidata.org/entity/Q19595707,http://www.wikidata.org/entity/Q142
2,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Adler, Jankel",http://www.wikidata.org/entity/Q36,http://www.wikidata.org/entity/Q145
3,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Adzak, Roy",http://www.wikidata.org/entity/Q145,http://www.wikidata.org/entity/Q142
4,https://w3id.org/FAIR-course-UT/2025-2026/grou...,Afro,http://www.wikidata.org/entity/Q38,http://www.wikidata.org/entity/Q39
5,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Agar, Eileen",http://www.wikidata.org/entity/Q414,http://www.wikidata.org/entity/Q145
6,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Agasse, Jacques Laurent",http://www.wikidata.org/entity/Q39,http://www.wikidata.org/entity/Q145
7,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Albers, Anni",http://www.wikidata.org/entity/Q183,http://www.wikidata.org/entity/Q30
8,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Albers, Josef",http://www.wikidata.org/entity/Q183,http://www.wikidata.org/entity/Q30
9,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Allen, George Warner",http://www.wikidata.org/entity/Q142,http://www.wikidata.org/entity/Q145


In [11]:
run("queries/cq4-living-at-snapshot.rq")

queries/cq4-living-at-snapshot.rq: 1232 rows


,artist,name,birth
0,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Abakanowicz, Magdalena",1930
1,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Abrahams, Ivor",1935
2,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Abts, Tomma",1967
3,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Acconci, Vito",1940
4,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Ackroyd, Norman",1938
5,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Ahtila, Eija-Liisa",1959
6,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Ai, Weiwei",1957
7,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Aksel, Erdag",1953
8,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Al-Ani, Jananne",1966
9,https://w3id.org/FAIR-course-UT/2025-2026/grou...,"Alechinsky, Pierre",1927


In [12]:
run("queries/cq5-wikidata-and-tate-page.rq")

queries/cq5-wikidata-and-tate-page.rq: 1 rows


,artist,wikidata,tatePage
0,https://w3id.org/FAIR-course-UT/2025-2026/grou...,http://www.wikidata.org/entity/Q259538,http://www.tate.org.uk/art/artists/edwin-austi...
